Классификация: IC50 > медианы выборки
Медиана IC50 ≈ 46.59 mM — бинарная классификация (0/1).
Модели: LogisticRegression, SVM, RandomForest, GradientBoosting.
Метрики: Accuracy, ROC-AUC, F1.

In [1]:
import os
os.makedirs("plots", exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, RocCurveDisplay, ConfusionMatrixDisplay
)
from sklearn.feature_selection import VarianceThreshold
import warnings

warnings.filterwarnings("ignore")

## Данные

In [2]:
df = pd.read_excel("data/data.xlsx", index_col=0)
targets = ["IC50, mM", "CC50, mM", "SI"]
features = [c for c in df.columns if c not in targets]

X = df[features]
ic50_median = df["IC50, mM"].median()
y = (df["IC50, mM"] > ic50_median).astype(int)
print(f"Медиана IC50 = {ic50_median:.4f} mM")
print(f"Класс 1 (IC50 > медиана): {y.sum()} ({y.mean()*100:.1f}%)")
print(f"Класс 0 (IC50 ≤ медиана): {(1-y).sum()} ({(1-y).mean()*100:.1f}%)")

selector = VarianceThreshold(threshold=0.0)
X = X.fillna(X.median())
X_sel = selector.fit_transform(X)
sel_features = [features[i] for i in selector.get_support(indices=True)]
X = pd.DataFrame(X_sel, columns=sel_features)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def cv_scores(model, X, y, cv):
    acc = cross_val_score(model, X, y, cv=cv, scoring="accuracy").mean()
    auc = cross_val_score(model, X, y, cv=cv, scoring="roc_auc").mean()
    f1  = cross_val_score(model, X, y, cv=cv, scoring="f1").mean()
    return acc, auc, f1

Медиана IC50 = 46.5852 mM
Класс 1 (IC50 > медиана): 500 (50.0%)
Класс 0 (IC50 ≤ медиана): 501 (50.0%)


## Базовые модели

In [3]:
print("\n── Базовые модели ──")
base_models = {
    "LogisticRegression": Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=1000))]),
    "SVM (RBF)":          Pipeline([("sc", StandardScaler()), ("m", SVC(probability=True))]),
    "RandomForest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
}
results = {}
for name, model in base_models.items():
    acc, auc, f1 = cv_scores(model, X, y, cv)
    results[name] = {"Accuracy": acc, "ROC-AUC": auc, "F1": f1}
    print(f"  {name:22s}: Acc={acc:.3f}  AUC={auc:.3f}  F1={f1:.3f}")


── Базовые модели ──
  LogisticRegression    : Acc=0.680  AUC=0.746  F1=0.681
  SVM (RBF)             : Acc=0.707  AUC=0.776  F1=0.709
  RandomForest          : Acc=0.716  AUC=0.778  F1=0.707
  GradientBoosting      : Acc=0.719  AUC=0.787  F1=0.711


## GridSearchCV

In [4]:
print("\n── GridSearchCV: LogisticRegression ──")
gs_lr = GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=2000))]),
    {"m__C": [0.01, 0.1, 1, 10, 100], "m__solver": ["lbfgs", "liblinear"]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_lr.fit(X, y)
print(f"  Best: {gs_lr.best_params_}  AUC={gs_lr.best_score_:.4f}")

print("\n── GridSearchCV: RandomForest ──")
gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {"n_estimators": [100, 200], "max_depth": [None, 10, 20], "min_samples_split": [2, 5]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_rf.fit(X, y)
print(f"  Best: {gs_rf.best_params_}  AUC={gs_rf.best_score_:.4f}")

print("\n── GridSearchCV: GradientBoosting ──")
gs_gb = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1, 0.2], "max_depth": [3, 5]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_gb.fit(X, y)
print(f"  Best: {gs_gb.best_params_}  AUC={gs_gb.best_score_:.4f}")


── GridSearchCV: LogisticRegression ──
  Best: {'m__C': 0.01, 'm__solver': 'lbfgs'}  AUC=0.7546

── GridSearchCV: RandomForest ──
  Best: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}  AUC=0.7857

── GridSearchCV: GradientBoosting ──
  Best: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}  AUC=0.7946


## Итог

In [5]:
print("\n── Итоговое сравнение ROC-AUC (tuned) ──")
tuned_auc = {
    "LogisticRegression": gs_lr.best_score_,
    "RandomForest":       gs_rf.best_score_,
    "GradientBoosting":   gs_gb.best_score_,
}
for name, auc in sorted(tuned_auc.items(), key=lambda x: -x[1]):
    print(f"  {name:22s}: AUC = {auc:.4f}")


── Итоговое сравнение ROC-AUC (tuned) ──
  GradientBoosting      : AUC = 0.7946
  RandomForest          : AUC = 0.7857
  LogisticRegression    : AUC = 0.7546


## ROC-кривые

In [6]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, gs in [("LogisticReg", gs_lr), ("RandomForest", gs_rf), ("GradBoosting", gs_gb)]:
    RocCurveDisplay.from_estimator(gs.best_estimator_, X, y, ax=ax, name=name)
ax.set_title("ROC-кривые (IC50 > медианы)")
plt.tight_layout()
plt.savefig("plots/cls_ic50_roc.png")
plt.close()
print("\nСохранён график: cls_ic50_roc.png")


Сохранён график: cls_ic50_roc.png


## Матрица ошибок лучшей модели

In [7]:
best_gs = max([gs_rf, gs_gb, gs_lr], key=lambda g: g.best_score_)
y_pred = best_gs.predict(X)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y, y_pred, ax=ax, colorbar=False)
ax.set_title("Матрица ошибок (лучшая модель, IC50 > медианы)")
plt.tight_layout()
plt.savefig("plots/cls_ic50_cm.png")
plt.close()
print("Сохранён график: cls_ic50_cm.png")

print("\nClassification Report (лучшая модель):")
print(classification_report(y, y_pred, target_names=["IC50 ≤ медианы", "IC50 > медианы"]))


Сохранён график: cls_ic50_cm.png

Classification Report (лучшая модель):
                precision    recall  f1-score   support

IC50 ≤ медианы       0.85      0.86      0.85       501
IC50 > медианы       0.86      0.85      0.85       500

      accuracy                           0.85      1001
     macro avg       0.85      0.85      0.85      1001
  weighted avg       0.85      0.85      0.85      1001



## Итоговые модели

- Выборка сбалансирована: 501 соединение с хорошей активностью (IC50 ≤ 46.6 мМ) и 500 с плохой активностью (IC50 > медианы).
- Базовые модели показывают умеренное качество: точность (Accuracy) около 70-72%, AUC в районе 0.75-0.79.
- Лучшая базовая модель — GradientBoosting с AUC=0.787 и Accuracy=0.719.
- Настройка гиперпараметров улучшает все модели. Лучший результат достигается у GradientBoosting с параметрами (learning_rate=0.05, max_depth=3, n_estimators=100) — AUC=0.7946.
- RandomForest после настройки даёт AUC=0.7857, LogisticRegression — AUC=0.7546.
- Разрыв между логистической регрессией (AUC=0.75) и градиентным бустингом (AUC=0.79) составляет 0.04 — умеренный, нелинейность присутствует, но не доминирует.
- Итоговое качество (AUC≈0.79) соответствует «приемлемому» уровню, но не отличному. Модель может различить два класса примерно в 79% случаев (вероятность правильно ранжировать пару «активное-неактивное»).
- По матрице ошибок (из графика) видно, сколько соединений классифицировано неверно, что позволяет оценить баланс между ложноположительными и ложноотрицательными ошибками.